<a href="https://colab.research.google.com/github/Andy-zheng98/-/blob/main/%E7%9D%A1%E7%9C%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '没有检测到 GPU，请先在“更改运行时类型”中选择 GPU'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('显存 GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

Wed Aug  5 03:03:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, '没有检测到 ZIP 文件，请重新运行本单元格并上传训练包'
ZIP_PATH = '/content/' + zip_names[0]
print('已上传:', ZIP_PATH)

Saving chatglm3_colab_lora_package.zip to chatglm3_colab_lora_package.zip
已上传: /content/chatglm3_colab_lora_package.zip


In [ ]:
import os, zipfile
WORK_DIR = '/content/chatglm3_colab_work'
os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    archive.extractall(WORK_DIR)
os.chdir(WORK_DIR)
required = ['data', 'configs', 'finetune_hf.py', 'requirements-single-gpu.txt']
missing = [name for name in required if not os.path.exists(name)]
assert not missing, f'解压后缺少文件: {missing}'
print('当前目录:', os.getcwd())
print('解压成功:', sorted(os.listdir()))

当前目录: /content/chatglm3_colab_work
解压成功: ['CHATGLM3_CODE_LICENSE', 'README.md', 'configs', 'data', 'dataset_stats.json', 'finetune_hf.py', 'inference_hf.py', 'requirements-single-gpu.txt', 'run_train.sh', 'verify_dataset.py']


In [ ]:
!pip install -q -r requirements-single-gpu.txt
print('依赖安装完成')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.6/416.6 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 require

In [ ]:
!python verify_dataset.py

OK: train=440, dev=60, total=500, group leakage=0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
config_path = Path('configs/lora_sleep.yaml')
text = config_path.read_text(encoding='utf-8')
text = text.replace('output_dir: ./output/chatglm3-sleep-lora', 'output_dir: /content/drive/MyDrive/chatglm3-sleep-lora')
config_path.write_text(text, encoding='utf-8')
print('模型将保存到 /content/drive/MyDrive/chatglm3-sleep-lora')

模型将保存到 /content/drive/MyDrive/chatglm3-sleep-lora


In [ ]:
!python finetune_hf.py data zai-org/chatglm3-6b configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards:  71% 5/7 [00:39<00:15,  7.90s/it]^C


In [ ]:
!python /content/chatglm3_colab_work/inference_hf.py /content/drive/MyDrive/chatglm3-sleep-lora --prompt '我最近晚上总是睡不着，而且很焦虑，应该怎么办？'

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:398 in     │
│ cached_file                                                                  │
│                                                                              │
│    395 │   user_agent = http_user_agent(user_agent)                          │
│    396 │   try:                                                              │
│    397 │   │   # Load from URL or cache if already cached                    │
│ ❱  398 │   │   resolved_file = hf_hub_download(                              │
│    399 │   │   │   path_or_repo_id,                                          │
│    400 │   │   │   filename,                                                 │
│    401 │   │   │   subfolder=None if len(subfolder) == 0 else subfolder,     │
│                                                                              │
│ /usr/local/lib/python3.12/

In [ ]:
from pathlib import Path

search_roots = [
    Path("/content/drive/MyDrive/chatglm3-sleep-lora"),
    Path("/content/chatglm3_colab_work/output/chatglm3-sleep-lora"),
]

adapter_files = []

for root in search_roots:
    print("检查目录：", root, "存在：", root.exists())
    if root.exists():
        adapter_files.extend(root.rglob("adapter_config.json"))

print("\n找到的适配器：")

for path in adapter_files:
    print(path)

检查目录： /content/drive/MyDrive/chatglm3-sleep-lora 存在： False
检查目录： /content/chatglm3_colab_work/output/chatglm3-sleep-lora 存在： False

找到的适配器：


In [ ]:
!cp -r \
  /content/chatglm3_colab_work/output/chatglm3-sleep-lora \
  /content/drive/MyDrive/

cp: cannot stat '/content/chatglm3_colab_work/output/chatglm3-sleep-lora': No such file or directory


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import os

print("Drive 是否挂载：", os.path.exists("/content/drive/MyDrive"))
print(
    "模型目录是否存在：",
    os.path.exists("/content/drive/MyDrive/chatglm3-sleep-lora")
)

Drive 是否挂载： True
模型目录是否存在： False


In [ ]:
from pathlib import Path
import shutil

possible_roots = [
    Path("/content/output/chatglm3-sleep-lora"),
    Path("/content/chatglm3_colab_work/output/chatglm3-sleep-lora"),
    Path("/content/chatglm3-sleep-lora"),
]

adapter_files = []

for root in possible_roots:
    print("检查目录：", root, "存在：", root.exists())

    if root.exists():
        found = list(root.rglob("adapter_config.json"))
        adapter_files.extend(found)

        for file in found:
            print("找到适配器：", file)

if adapter_files:
    latest_adapter = max(
        (file.parent for file in adapter_files),
        key=lambda path: path.stat().st_mtime
    )

    print("\n最新适配器：", latest_adapter)

    drive_output = Path(
        "/content/drive/MyDrive/chatglm3-sleep-lora"
    )

    source_root = next(
        root for root in possible_roots
        if root.exists() and latest_adapter.is_relative_to(root)
    )

    shutil.copytree(
        source_root,
        drive_output,
        dirs_exist_ok=True
    )

    print("已复制到 Google Drive：", drive_output)

else:
    print("\n没有找到训练适配器，需要重新训练。")

检查目录： /content/output/chatglm3-sleep-lora 存在： False
检查目录： /content/chatglm3_colab_work/output/chatglm3-sleep-lora 存在： False
检查目录： /content/chatglm3-sleep-lora 存在： False

没有找到训练适配器，需要重新训练。


In [ ]:
from pathlib import Path

work_dir = Path("/content/chatglm3_colab_work")

print("工作目录：", work_dir.exists())
print("训练脚本：", (work_dir / "finetune_hf.py").exists())
print("训练数据：", (work_dir / "data/train.json").exists())
print("配置文件：", (work_dir / "configs/lora_sleep.yaml").exists())

工作目录： True
训练脚本： True
训练数据： True
配置文件： True


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import os

assert os.path.exists("/content/drive/MyDrive")
print("Google Drive 已挂载")

Google Drive 已挂载


In [ ]:
from pathlib import Path

config_path = Path(
    "/content/chatglm3_colab_work/configs/lora_sleep.yaml"
)

assert config_path.exists(), "配置文件不存在，请重新上传并解压训练包"

config_text = """
data_config:
  train_file: train.json
  val_file: dev.json
  test_file: dev.json
  num_proc: 4

max_input_length: 512
max_output_length: 512

training_args:
  output_dir: /content/drive/MyDrive/chatglm3-sleep-lora
  num_train_epochs: 3
  learning_rate: 5e-5
  per_device_train_batch_size: 1
  gradient_accumulation_steps: 8
  dataloader_num_workers: 4
  remove_unused_columns: false

  save_strategy: steps
  save_steps: 25
  save_total_limit: 3

  logging_strategy: steps
  logging_steps: 10

  per_device_eval_batch_size: 1
  evaluation_strategy: epoch
  predict_with_generate: true

  generation_config:
    max_new_tokens: 512

  use_cpu: false

peft_config:
  peft_type: LORA
  task_type: CAUSAL_LM
  r: 8
  lora_alpha: 32
  lora_dropout: 0.1
""".strip() + "\n"

config_path.write_text(config_text, encoding="utf-8")

print(config_path.read_text(encoding="utf-8"))

data_config:
  train_file: train.json
  val_file: dev.json
  test_file: dev.json
  num_proc: 4

max_input_length: 512
max_output_length: 512

training_args:
  output_dir: /content/drive/MyDrive/chatglm3-sleep-lora
  num_train_epochs: 3
  learning_rate: 5e-5
  per_device_train_batch_size: 1
  gradient_accumulation_steps: 8
  dataloader_num_workers: 4
  remove_unused_columns: false

  save_strategy: steps
  save_steps: 25
  save_total_limit: 3

  logging_strategy: steps
  logging_steps: 10

  per_device_eval_batch_size: 1
  evaluation_strategy: epoch
  predict_with_generate: true

  generation_config:
    max_new_tokens: 512

  use_cpu: false

peft_config:
  peft_type: LORA
  task_type: CAUSAL_LM
  r: 8
  lora_alpha: 32
  lora_dropout: 0.1



In [ ]:
!python /content/chatglm3_colab_work/verify_dataset.py

OK: train=440, dev=60, total=500, group leakage=0


In [ ]:
!python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards:  71% 5/7 [00:39<00:15,  7.90s/it]^C


In [ ]:
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/chatglm3-sleep-lora"
)

print("输出目录存在：", output_dir.exists())

if output_dir.exists():
    for path in output_dir.rglob("adapter_config.json"):
        print("已保存适配器：", path)

输出目录存在： False


In [ ]:
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/chatglm3-sleep-lora"
)

output_dir.mkdir(parents=True, exist_ok=True)

test_file = output_dir / "write_test.txt"
test_file.write_text("Google Drive write test", encoding="utf-8")

print("输出目录存在：", output_dir.exists())
print("测试文件存在：", test_file.exists())
print("输出目录：", output_dir)

输出目录存在： True
测试文件存在： True
输出目录： /content/drive/MyDrive/chatglm3-sleep-lora


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path

config_path = Path(
    "/content/chatglm3_colab_work/configs/lora_sleep.yaml"
)

assert config_path.exists(), "训练配置不存在，需要重新上传并解压训练包"

config_text = config_path.read_text(encoding="utf-8")

for line in config_text.splitlines():
    if any(key in line for key in [
        "output_dir",
        "save_strategy",
        "save_steps",
        "save_total_limit",
    ]):
        print(line)

  output_dir: /content/drive/MyDrive/chatglm3-sleep-lora
  save_strategy: steps
  save_steps: 25
  save_total_limit: 3


In [ ]:
from pathlib import Path

required_files = [
    "/content/chatglm3_colab_work/finetune_hf.py",
    "/content/chatglm3_colab_work/data/train.json",
    "/content/chatglm3_colab_work/data/dev.json",
    "/content/chatglm3_colab_work/configs/lora_sleep.yaml",
]

for file in required_files:
    print(Path(file).exists(), file)

True /content/chatglm3_colab_work/finetune_hf.py
True /content/chatglm3_colab_work/data/train.json
True /content/chatglm3_colab_work/data/dev.json
True /content/chatglm3_colab_work/configs/lora_sleep.yaml


In [ ]:
!python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards:  71% 5/7 [00:39<00:16,  8.02s/it]^C


In [ ]:
!python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards:  71% 5/7 [00:39<00:16,  8.01s/it]^C


In [ ]:
import torch

print("CUDA 可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))
    print(
        "显存容量 GB：",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            1
        )
    )

CUDA 可用： True
GPU： Tesla T4
显存容量 GB： 14.6


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("显存缓存已清理")

显存缓存已清理


In [ ]:
!python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards:  71% 5/7 [00:39<00:15,  7.93s/it]^C


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("显存已清理")


显存已清理


In [ ]:
!python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards:  71% 5/7 [00:39<00:15,  7.92s/it]^C


In [ ]:
import time

print("开始测试，请等待 60 秒，不要点击停止")
time.sleep(60)
print("60 秒测试完成")

开始测试，请等待 60 秒，不要点击停止
60 秒测试完成


In [ ]:
!python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards:  71% 5/7 [00:39<00:15,  7.99s/it]^C


In [ ]:
!free -h
!nvidia-smi

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi        10Gi       2.0Mi       637Mi        11Gi
Swap:             0B          0B          0B
Wed Aug  5 06:59:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8        

In [ ]:
from pathlib import Path

script_path = Path(
    "/content/chatglm3_colab_work/finetune_hf.py"
)

assert script_path.exists(), "finetune_hf.py 不存在，请重新上传并解压训练包"

text = script_path.read_text(encoding="utf-8")

old_code = """empty_init=False,
                use_cache=False
"""

new_code = """empty_init=False,
                use_cache=False,
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                device_map="auto",
"""

replace_count = text.count(old_code)

print("找到需要修改的位置：", replace_count)

assert replace_count >= 1, "没有找到目标代码，可能脚本已经修改过"

text = text.replace(old_code, new_code)

script_path.write_text(text, encoding="utf-8")

print("低内存加载补丁已经写入")

找到需要修改的位置： 1
低内存加载补丁已经写入


In [ ]:
from pathlib import Path

script_path = Path(
    "/content/chatglm3_colab_work/finetune_hf.py"
)

text = script_path.read_text(encoding="utf-8")

for line in text.splitlines():
    if any(key in line for key in [
        "torch_dtype",
        "low_cpu_mem_usage",
        "device_map",
    ]):
        print(line)

                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                device_map="auto",


In [ ]:
import os

print(
    "Google Drive 已挂载：",
    os.path.exists("/content/drive/MyDrive")
)

Google Drive 已挂载： True


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU 缓存已清理")

GPU 缓存已清理


In [ ]:
!free -h
!nvidia-smi

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi        10Gi       2.0Mi       671Mi        11Gi
Swap:             0B          0B          0B
Wed Aug  5 07:01:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8        

In [ ]:
from pathlib import Path
import re

config_path = Path(
    "/content/chatglm3_colab_work/configs/lora_sleep.yaml"
)

text = config_path.read_text(encoding="utf-8")

text = re.sub(
    r"max_input_length:\s*\d+",
    "max_input_length: 256",
    text,
)

text = re.sub(
    r"max_output_length:\s*\d+",
    "max_output_length: 384",
    text,
)

config_path.write_text(text, encoding="utf-8")

for line in text.splitlines():
    if any(key in line for key in [
        "max_input_length",
        "max_output_length",
        "output_dir",
        "save_steps",
    ]):
        print(line)

max_input_length: 256
max_output_length: 384
  output_dir: /content/drive/MyDrive/chatglm3-sleep-lora
  save_steps: 25


In [ ]:
from pathlib import Path

script_path = Path(
    "/content/chatglm3_colab_work/finetune_hf.py"
)

text = script_path.read_text(encoding="utf-8")

required = [
    "torch_dtype=torch.float16",
    "low_cpu_mem_usage=True",
    'device_map="auto"',
]

for item in required:
    print(item, "→", item in text)

torch_dtype=torch.float16 → True
low_cpu_mem_usage=True → True
device_map="auto" → True


In [ ]:
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/chatglm3-sleep-lora"
)

output_dir.mkdir(parents=True, exist_ok=True)

test_file = output_dir / "write_test.txt"
test_file.write_text("OK", encoding="utf-8")

print("目录存在：", output_dir.exists())
print("能够写入：", test_file.exists())

目录存在： True
能够写入： True


In [ ]:
!python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards: 100% 7/7 [00:51<00:00,  7.40s/it]
trainable params: 1,949,696 || all params: 6,245,533,696 || trainable%: 0.031217444255383614
--> Model

--> model has 1.949696M params

train_dataset: Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 440
})
Map (num_proc=4): 100% 60/60 [00:00<00:00, 161.90 examples/s]
val_dataset: Dataset({
    features: ['input_ids', 'output_ids'],
    num_rows: 60
})
test_dataset: Dataset({
    features: ['input_ids', 'output_ids'],
    num_rows: 60
})
--> Sanity check
           '

In [ ]:
from pathlib import Path

config_path = Path(
    "/content/chatglm3_colab_work/configs/lora_sleep.yaml"
)

assert config_path.exists(), "配置文件不存在"

config_text = """
data_config:
  train_file: train.json
  val_file: dev.json
  test_file: dev.json
  num_proc: 2

max_input_length: 128
max_output_length: 256

training_args:
  output_dir: /content/drive/MyDrive/chatglm3-sleep-lora

  num_train_epochs: 3
  learning_rate: 5e-5

  per_device_train_batch_size: 1
  gradient_accumulation_steps: 8
  dataloader_num_workers: 2
  remove_unused_columns: false

  save_strategy: steps
  save_steps: 10
  save_total_limit: 3

  logging_strategy: steps
  logging_steps: 5
  report_to: none

  per_device_eval_batch_size: 1
  evaluation_strategy: epoch
  predict_with_generate: true

  generation_config:
    max_new_tokens: 256

  use_cpu: false

peft_config:
  peft_type: LORA
  task_type: CAUSAL_LM
  r: 8
  lora_alpha: 32
  lora_dropout: 0.1
""".strip() + "\n"

config_path.write_text(config_text, encoding="utf-8")

print(config_path.read_text(encoding="utf-8"))

data_config:
  train_file: train.json
  val_file: dev.json
  test_file: dev.json
  num_proc: 2

max_input_length: 128
max_output_length: 256

training_args:
  output_dir: /content/drive/MyDrive/chatglm3-sleep-lora

  num_train_epochs: 3
  learning_rate: 5e-5

  per_device_train_batch_size: 1
  gradient_accumulation_steps: 8
  dataloader_num_workers: 2
  remove_unused_columns: false

  save_strategy: steps
  save_steps: 10
  save_total_limit: 3

  logging_strategy: steps
  logging_steps: 5
  report_to: none

  per_device_eval_batch_size: 1
  evaluation_strategy: epoch
  predict_with_generate: true

  generation_config:
    max_new_tokens: 256

  use_cpu: false

peft_config:
  peft_type: LORA
  task_type: CAUSAL_LM
  r: 8
  lora_alpha: 32
  lora_dropout: 0.1



In [ ]:
from pathlib import Path

script_path = Path(
    "/content/chatglm3_colab_work/finetune_hf.py"
)

text = script_path.read_text(encoding="utf-8")

for item in [
    "torch_dtype=torch.float16",
    "low_cpu_mem_usage=True",
    'device_map="auto"',
]:
    print(item, "→", item in text)

torch_dtype=torch.float16 → True
low_cpu_mem_usage=True → True
device_map="auto" → True


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

print("PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])
print("WANDB_DISABLED =", os.environ["WANDB_DISABLED"])

PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True
WANDB_DISABLED = true


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU 缓存已清理")

GPU 缓存已清理


In [ ]:
!nvidia-smi

Wed Aug  5 07:14:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/chatglm3-sleep-lora"
)

output_dir.mkdir(parents=True, exist_ok=True)

test_file = output_dir / "write_test.txt"
test_file.write_text("OK", encoding="utf-8")

print("目录存在：", output_dir.exists())
print("能够写入：", test_file.exists())

目录存在： True
能够写入： True


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
WANDB_DISABLED=true \
python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards: 100% 7/7 [00:51<00:00,  7.42s/it]
trainable params: 1,949,696 || all params: 6,245,533,696 || trainable%: 0.031217444255383614
--> Model

--> model has 1.949696M params

Map (num_proc=2): 100% 440/440 [00:00<00:00, 1350.10 examples/s]
train_dataset: Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 440
})
Map (num_proc=2): 100% 60/60 [00:00<00:00, 264.52 examples/s]
val_dataset: Dataset({
    features: ['input_ids', 'output_ids'],
    num_rows: 43
})
Map (num_proc=2): 100% 60/60 [00:00<00:00, 264.38 e

In [ ]:
from pathlib import Path

script_path = Path(
    "/content/chatglm3_colab_work/finetune_hf.py"
)

assert script_path.exists(), "训练脚本不存在"

text = script_path.read_text(encoding="utf-8")

old_code = """
eval_dataset=val_dataset.select(list(range(50))),
""".strip()

new_code = """
eval_dataset=val_dataset.select(
    list(range(min(50, len(val_dataset))))
),
""".strip()

print("找到旧代码：", old_code in text)

assert old_code in text, "没有找到需要修改的代码，可能已经修复"

text = text.replace(old_code, new_code)

script_path.write_text(text, encoding="utf-8")

print("验证集数量修复完成")

找到旧代码： True
验证集数量修复完成


In [ ]:
from pathlib import Path

script_path = Path(
    "/content/chatglm3_colab_work/finetune_hf.py"
)

text = script_path.read_text(encoding="utf-8")

print(
    "动态验证集选择已启用：",
    "min(50, len(val_dataset))" in text
)

动态验证集选择已启用： True


In [ ]:
for item in [
    "torch_dtype=torch.float16",
    "low_cpu_mem_usage=True",
    'device_map="auto"',
]:
    print(item, "→", item in text)

torch_dtype=torch.float16 → True
low_cpu_mem_usage=True → True
device_map="auto" → True


In [ ]:
from pathlib import Path

config_path = Path(
    "/content/chatglm3_colab_work/configs/lora_sleep.yaml"
)

config_text = config_path.read_text(encoding="utf-8")

for line in config_text.splitlines():
    if any(key in line for key in [
        "max_input_length",
        "max_output_length",
        "output_dir",
        "save_steps",
        "dataloader_num_workers",
    ]):
        print(line)

max_input_length: 128
max_output_length: 256
  output_dir: /content/drive/MyDrive/chatglm3-sleep-lora
  dataloader_num_workers: 2
  save_steps: 10


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("显存缓存已清理")

显存缓存已清理


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
WANDB_DISABLED=true \
python /content/chatglm3_colab_work/finetune_hf.py \
  /content/chatglm3_colab_work/data \
  zai-org/chatglm3-6b \
  /content/chatglm3_colab_work/configs/lora_sleep.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards: 100% 7/7 [00:52<00:00,  7.46s/it]
trainable params: 1,949,696 || all params: 6,245,533,696 || trainable%: 0.031217444255383614
--> Model

--> model has 1.949696M params

train_dataset: Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 440
})
Map (num_proc=2): 100% 60/60 [00:00<00:00, 238.27 examples/s]
val_dataset: Dataset({
    features: ['input_ids', 'output_ids'],
    num_rows: 43
})
test_dataset: Dataset({
    features: ['input_ids', 'output_ids'],
    num_rows: 43
})
--> Sanity check
           '

In [ ]:
import os

output_dir = "/content/drive/MyDrive/chatglm3-sleep-lora"

print("目录是否存在：", os.path.exists(output_dir))

if os.path.exists(output_dir):
    print("目录内容：")
    for name in os.listdir(output_dir):
        print(" -", name)

目录是否存在： True
目录内容：
 - write_test.txt
 - runs
 - checkpoint-140
 - checkpoint-150
 - checkpoint-160


In [ ]:
import os
import glob

output_dir = "/content/drive/MyDrive/chatglm3-sleep-lora"

adapter_files = (
    glob.glob(output_dir + "/adapter_model.bin") +
    glob.glob(output_dir + "/adapter_model.safetensors") +
    glob.glob(output_dir + "/checkpoint-*/adapter_model.bin") +
    glob.glob(output_dir + "/checkpoint-*/adapter_model.safetensors")
)

checkpoints = glob.glob(output_dir + "/checkpoint-*")

print("找到的适配器文件：")
for path in adapter_files:
    print("✅", path)

print("\n找到的检查点：")
for path in checkpoints:
    print("✅", path)

if adapter_files:
    print("\n🎉 微调模型已经保存成功，可以进行推理测试。")
else:
    print("\n❌ 没找到适配器文件，请不要关闭运行时，把输出结果发给我。")

找到的适配器文件：
✅ /content/drive/MyDrive/chatglm3-sleep-lora/checkpoint-140/adapter_model.safetensors
✅ /content/drive/MyDrive/chatglm3-sleep-lora/checkpoint-150/adapter_model.safetensors
✅ /content/drive/MyDrive/chatglm3-sleep-lora/checkpoint-160/adapter_model.safetensors

找到的检查点：
✅ /content/drive/MyDrive/chatglm3-sleep-lora/checkpoint-140
✅ /content/drive/MyDrive/chatglm3-sleep-lora/checkpoint-150
✅ /content/drive/MyDrive/chatglm3-sleep-lora/checkpoint-160

🎉 微调模型已经保存成功，可以进行推理测试。


In [ ]:
import gc
import torch

for name in ["trainer", "model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

!nvidia-smi

Wed Aug  5 08:41:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q \
  "transformers==4.40.2" \
  "peft==0.10.0" \
  "accelerate==0.30.1" \
  "sentencepiece" \
  "protobuf"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 9.4 MB/s eta 0:00:00
